In [2]:
# !pip install swebench

In [4]:
# !docker ps

In [1]:
import json

dummy_data = [
    {
        "instance_id": "django__django-11133",
        "model_patch": "diff --git a/django/utils/http.py b/django/utils/http.py\nindex 5a5749b..7c7e1f4 100\n--- a/django/utils/http.py\n+++ b/django/utils/http.py\n@@ -1 +1 @@\n-# Dummy comment\n+# Fix applied",
        "model_name_or_path": "dummy_model_v1"
    },
    {
        "instance_id": "scikit-learn__scikit-learn-10297",
        "model_patch": "", # Empty patch
        "model_name_or_path": "dummy_model_v1"
    }
]

with open("dummy_predictions.jsonl", "w") as f:
    for entry in dummy_data:
        f.write(json.dumps(entry) + "\n")

print("✅ Created dummy_predictions.jsonl")

✅ Created dummy_predictions.jsonl


In [5]:
import json

dummy_data = [
    {
        "instance_id": "django__django-11133",
        "model_patch": (
            "diff --git a/django/http/response.py b/django/http/response.py\n"
            "--- a/django/http/response.py\n"
            "+++ b/django/http/response.py\n"
            "@@ -229,7 +229,7 @@\n"
            "         # Handle string types -- we can't rely on force_bytes here because:\n"
            "         # - Python attempts str conversion first\n"
            "         # - when self._charset != 'utf-8' it re-encodes the content\n"
            "-        if isinstance(value, bytes):\n"
            "+        if isinstance(value, (bytes, memoryview)):\n"
            "             return bytes(value)\n"
            "         if isinstance(value, str):\n"
            "             return bytes(value.encode(self.charset))\n"
        ),
        "model_name_or_path": "gold_patch_test"
    }
]

with open("dummy_predictions.jsonl", "w") as f:
    for entry in dummy_data:
        f.write(json.dumps(entry) + "\n")

print("✅ Created dummy_predictions.jsonl with Gold Patch")

✅ Created dummy_predictions.jsonl with Gold Patch


In [6]:
!python -m swebench.harness.run_evaluation \
    --dataset_name princeton-nlp/SWE-bench_Verified \
    --predictions_path dummy_predictions.jsonl \
    --max_workers 1 \
    --run_id dummy_test_run \
    --cache_level env

<frozen runpy>:128: RuntimeWarning: 'swebench.harness.run_evaluation' found in sys.modules after import of package 'swebench.harness', but prior to execution of 'swebench.harness.run_evaluation'; this may result in unpredictable behaviour
2026-04-08 10:57:07,255 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/princeton-nlp/SWE-bench_Verified/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-04-08 10:57:07,317 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/princeton-nlp/SWE-bench_Verified/c104f840cc67f8b6eec6f759ebc8b2693d585d4a/README.md "HTTP/1.1 200 OK"
2026-04-08 10:57:07,449 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/princeton-nlp/SWE-bench_Verified/resolve/c104f840cc67f8b6eec6f759ebc8b2693d585d4a/SWE-bench_Verified.py "HTTP/1.1 404 Not Found"
2026-04-08 10:57:11,206 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/datasets/princeton-nlp/SWE-bench_Verified/revision/c104f840c

In [ ]:
import os
import json

# The harness typically saves results in a folder named after your run_id
report_path = "outputs/dummy_test_run/report.json"

if os.path.exists(report_path):
    with open(report_path, "r") as f:
        results = json.load(f)
    
    print("--- EVALUATION SUMMARY ---")
    print(f"Total Instances: {results.get('total', 0)}")
    print(f"Resolved (Passed): {len(results.get('resolved', []))}")
    print(f"Failed: {len(results.get('failed', []))}")
else:
    print("❌ Report not found. Check the console output for errors.")
